In [1]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [2]:
Base_Dir = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
train_dir = os.path.join(Base_Dir,"train")
test_dir = os.path.join(Base_Dir,"test")

In [3]:
data_transforms = {
    "train":transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),

    "test":transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

In [4]:
image_datasets = {
    "train":datasets.ImageFolder(train_dir,data_transforms['train']),
    "test":datasets.ImageFolder(test_dir,data_transforms["test"])
}

In [5]:
dataloaders = {x:DataLoader(image_datasets[x], batch_size=32, shuffle = True if x == "train" else False, num_workers=2) for x in ["train","test"]}

In [6]:
dataset_sizes = {x:len(image_datasets[x]) for x in ["train","test"]}
class_names = image_datasets["train"].classes

print(f" Classes detected:{class_names}")
print(f" Total training Images:{dataset_sizes["train"]}")
print(f" Total testing Images:{dataset_sizes["test"]}")

 Classes detected:['NORMAL', 'PNEUMONIA']
 Total training Images:5216
 Total testing Images:624


In [8]:
device = torch.device("cuda")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 150MB/s] 


In [9]:
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs,2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [11]:
def train_model(model, criterion, optimizer, num_epochs = 5):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-"*10)
        for phase in ["train","test"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _,preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == "train":
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item()*inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]
            print(f"{phase.capitalize()} Loss:{epoch_loss:.4f} Acc:{epoch_acc:.4f}")

            if phase =="test" and epoch_acc >best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

    time_elapsed = time.time() - since
    print(f"\n Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Test Accuracy: {best_acc:4f}")
    model.load_state_dict(best_model_wts)
    return model

trained_model = train_model(model, criterion, optimizer, num_epochs =5)

save_path = "/kaggle/working/pneumonia_resnet18.pth"
torch.save(trained_model.state_dict(), save_path)


Epoch 1/5
----------
Train Loss:0.1270 Acc:0.9532
Test Loss:0.4701 Acc:0.8462

Epoch 2/5
----------
Train Loss:0.0741 Acc:0.9732
Test Loss:1.1860 Acc:0.6987

Epoch 3/5
----------
Train Loss:0.0648 Acc:0.9787
Test Loss:0.5596 Acc:0.8205

Epoch 4/5
----------
Train Loss:0.0551 Acc:0.9797
Test Loss:0.3486 Acc:0.9022

Epoch 5/5
----------
Train Loss:0.0562 Acc:0.9806
Test Loss:0.9817 Acc:0.7740

 Training complete in 4m 37s
Best Test Accuracy: 0.902244
